# Part 8: The Capstone Project
## Content Creation Studio Workshop - Complete System Integration

Welcome to the final part of the Content Creation Studio Playbook! Over the past seven parts, you've learned all the fundamental patterns for building sophisticated AI agent systems:
- Basic agents with tools
- Custom function tools
- Agent teams and orchestration
- Sequential workflows
- Iterative quality loops
- Parallel execution
- Callbacks, context management, and memory

Now it's time to **put it all together** in a complete, autonomous Content Creation Studio that combines every pattern we've learned!

---

## The Content Creation Studio Playbook Series
- Part 1: Building Your First AI Agent 
- Part 2: Giving Your Agent Custom Tools 
- Part 3: Building Agent Teams with Agent-as-a-Tool 
- Part 4: Sequential Workflows & Design Patterns 
- Part 5: Building Iterative Workflows with LoopAgent 
- Part 6: Efficient Workflows with ParallelAgent 
- Part 7: Callbacks, Context & Memory 
- **Part 8: The Capstone Project** (You are here) 
- Part 9: Deploying to Vertex AI Agent Engine

---

## New Concepts in This Part

In Part 8, we'll introduce these **capstone-level ADK concepts**:

1. **Hierarchical Orchestration** - Master agents managing sub-workflows and specialists
2. **Composable Agent Systems** - Modular components that combine seamlessly
3. **End-to-End Autonomous Workflows** - Complete task execution with minimal intervention

Each concept will be clearly marked with when first introduced!

---

## The Capstone Architecture

### NEW CONCEPT: End-to-End Autonomous Workflows

> **What are End-to-End Autonomous Workflows?** 
> End-to-end autonomous workflows are complete, self-contained agent systems that can take a high-level user request and independently execute all necessary steps to completion with minimal human intervention. They combine multiple workflow patterns (sequential, loop, parallel) and specialist agents into a cohesive pipeline that handles the entire lifecycle of a task.
>
> **Key Characteristics**:
> - **Autonomous**: Executes complex multi-step processes without step-by-step human guidance
> - **Complete**: Handles entire workflow from input to final deliverable
> - **Adaptive**: Can handle errors, make decisions, and adjust approach
> - **Integrated**: Combines ALL workflow patterns (Sequential, Loop, Parallel)
>
> **The Complete Workflow**:
> ```
> User Request → Research → Draft → Improve (Loop) → Multi-Channel (Parallel) → Deliver
> ```
>
> **Benefits**:
> - Minimal human intervention required
> - Consistent quality through automated checks
> - Efficient through parallel execution
> - Scalable to handle increased workload
> - Reliable through clear error handling
>
> **Real-World Applications**:
> - Content creation (our example)
> - Data processing pipelines
> - Customer service automation
> - Research and analysis workflows
>
> **Reference**: [ADK Overview](https://google.github.io/adk-docs/)

---

Before we dive into the code, let's look at the high-level architecture. We will build a **Master Orchestrator** that uses `sub_agents` to transfer control to its team. Based on the user's request, it can trigger a complex, autonomous content workflow or switch to a simple content analyzer.

```
 ┌──────────────────┐
 │ Master │
 │ Orchestrator │
 │ + Callbacks │
 │ + Memory │
 └────────┬─────────┘
 │ (transfer_to_agent)
 ┌────────────┴────────────┐
 │ │
 
 ┌────────────────┐ ┌─────────────────┐
 │ Full Content │ │ Content │
 │ Workflow │ │ Analyzer │
 │ (sub_agent) │ │ (sub_agent) │
 └────────────────┘ └─────────────────┘
 │
 
 ┌────────────────┐
 │ 1. Sequential │ (Topic Research → Draft)
 └────────┬───────┘
 
 ┌────────────────┐
 │ 2. Loop │ (Quality Check → Improve)
 └────────┬───────┘
 
 ┌────────────────┐
 │ 3. Parallel │ (Blog + Social + Email + SEO)
 └────────────────┘
```

Each parallel agent stores its output via `output_key`, so results are accessible individually — no final packager agent needed. All inner events propagate to the Runner thanks to `sub_agents`.

This represents a complete, autonomous workflow that combines all the patterns from Parts 1-6!

## Setup: Install Google ADK

First, let's install the required library.

## Setup: Configure Your API Key

Before we can build agents, we need to authenticate with Google's Gemini API.

### How to Get Your API Key
1. Navigate to [Google AI Studio](https://aistudio.google.com)
2. Sign in with your Google Account
3. Click the "Get API key" button
4. Follow the prompts to create a project
5. Copy your API key

In [ ]:
import os
import json
import re
from typing import List
from getpass import getpass
from IPython.display import display, Markdown

# Prompt for API key securely
api_key = getpass('Enter your Google API Key: ')
os.environ['GOOGLE_API_KEY'] = api_key

print(" API Key configured successfully!")

## 1. Define All Custom Tools

First, we define all the custom Python functions that our various specialist agents will need throughout the workflow. These tools give our agents specialized capabilities for content analysis, quality checking, and workflow control.

In [ ]:
from google.adk.tools import ToolContext
import time
import logging
from typing import Optional, List
from google.adk.agents.callback_context import CallbackContext
from google.adk.models import LlmResponse, LlmRequest
from google.genai import types

# --- Content Analysis Tools ---

def count_words(text: str) -> int:
 """Counts the number of words in the provided text."""
 print(f" Tool: Counting words...")
 count = len(text.split())
 print(f" Result: {count} words")
 return count

def calculate_readability_score(text: str) -> dict:
 """Calculates a readability score (0-100, higher is easier to read)."""
 print(f" Tool: Calculating readability...")
 sentences = [s.strip() for s in text.split('.') if s.strip()]
 if not sentences:
 return {"score": 0, "grade": "Unable to calculate"}
 words = text.split()
 total_words = len(words)
 total_sentences = len(sentences)
 total_syllables = sum(count_syllables(word) for word in words)
 if total_words == 0 or total_sentences == 0:
 score = 0
 else:
 score = 206.835 - 1.015 * (total_words / total_sentences) - 84.6 * (total_syllables / total_words)
 score = max(0, min(100, score))
 grade = "Easy to read" if score >= 60 else ("Moderate" if score >= 50 else "Complex")
 result = {"score": round(score, 2), "grade": grade}
 print(f" Result: {result['score']} - {result['grade']}")
 return result

def count_syllables(word: str) -> int:
 """Helper function to estimate syllables in a word."""
 word = word.lower()
 vowels = "aeiouy"
 syllable_count = 0
 previous_was_vowel = False
 for char in word:
 is_vowel = char in vowels
 if is_vowel and not previous_was_vowel:
 syllable_count += 1
 previous_was_vowel = is_vowel
 if word.endswith('e'):
 syllable_count -= 1
 return max(1, syllable_count)

def generate_hashtags(text: str, count: int = 5) -> List[str]:
 """Generates relevant hashtags from text by extracting key terms."""
 print(f" Tool: Generating {count} hashtags...")
 stop_words = {
 'the', 'is', 'at', 'which', 'on', 'a', 'an', 'as', 'are', 'was', 'were',
 'been', 'be', 'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would',
 'could', 'should', 'may', 'might', 'must', 'can', 'of', 'to', 'for', 'in',
 'with', 'by', 'from', 'up', 'about', 'into', 'through', 'during', 'and',
 'or', 'but', 'if', 'then', 'than', 'so', 'this', 'that', 'these', 'those'
 }
 import re
 words = re.findall(r'\b[a-zA-Z]{4,}\b', text.lower())
 word_freq = {}
 for word in words:
 if word not in stop_words:
 word_freq[word] = word_freq.get(word, 0) + 1
 sorted_words = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)
 hashtags = [f"#{w.capitalize()}" for w, _ in sorted_words[:count]]
 print(f" Result: {', '.join(hashtags)}")
 return hashtags

# --- Quality Check Tool ---

def calculate_content_quality_score(
 word_count: int, readability_score: float, has_headings: bool, has_conclusion: bool
) -> dict:
 """Calculates overall content quality score based on multiple factors."""
 print(f" Tool: Calculating quality score...")
 word_score = 30 if word_count < 500 else (60 if word_count < 800 else (100 if word_count <= 2000 else 80))
 read_score = min(100, readability_score * 1.5) if readability_score > 0 else 40
 structure_score = (50 if has_headings else 0) + (50 if has_conclusion else 0)
 overall_score = (word_score * 0.3) + (read_score * 0.3) + (structure_score * 0.4)
 result = {"overall_score": round(overall_score, 2), "word_count": word_count, "meets_threshold": overall_score >= 70}
 print(f" Result: {result['overall_score']}/100 ({'MET' if result['meets_threshold'] else 'NOT MET'})")
 return result

# --- Loop Control ---
QUALITY_THRESHOLD_MET = "QUALITY_THRESHOLD_MET"

def exit_loop(tool_context: ToolContext):
 """Terminates the improvement loop when quality meets threshold."""
 print(f" Tool: Quality approved. Terminating loop...")
 tool_context.actions.escalate = True
 return {"result": "Quality threshold met. Content approved."}

# Callbacks - see Part 7 for detailed explanation of each callback type

BLOCKED_TOPICS = ["violence", "illegal", "harmful", "hate speech"]
_agent_start_times = {}

def before_agent_callback(callback_context: CallbackContext) -> Optional[types.Content]:
 """Logs agent start and tracks execution time."""
 agent_name = callback_context.agent_name
 _agent_start_times[agent_name] = time.time()
 print(f"\n{'─'*50}")
 print(f" AGENT START: {agent_name}")
 print(f"{'─'*50}")
 return None # Proceed normally

def after_agent_callback(callback_context: CallbackContext) -> Optional[types.Content]:
 """Logs agent completion with execution time."""
 agent_name = callback_context.agent_name
 elapsed = time.time() - _agent_start_times.pop(agent_name, time.time())
 print(f"\n{'─'*50}")
 print(f"■ AGENT DONE: {agent_name} ({elapsed:.1f}s)")
 print(f"{'─'*50}")
 return None

def before_model_callback(callback_context: CallbackContext, llm_request: LlmRequest) -> Optional[LlmResponse]:
 """Content safety guardrail — blocks prohibited topics before they reach the model."""
 if llm_request.contents:
 last = llm_request.contents[-1]
 if last.parts:
 text = (last.parts[0].text or "").lower()
 for topic in BLOCKED_TOPICS:
 if topic in text:
 print(f" GUARDRAIL: Blocked '{topic}'")
 return LlmResponse(
 content=types.Content(
 parts=[types.Part.from_text(text=f"I cannot generate content about '{topic}'.")],
 role="model"
 )
 )
 return None

def after_model_callback(callback_context: CallbackContext, llm_response: LlmResponse) -> Optional[LlmResponse]:
 """Logs model output word count."""
 if llm_response.content and llm_response.content.parts:
 text = llm_response.content.parts[0].text or ""
 print(f" Model output for {callback_context.agent_name}: ~{len(text.split())} words")
 return None

print(" All tools and callbacks defined!")

## 2. Define All Specialist Agents

Now we define our complete team of specialist agents. Each agent is an expert at a single job. This modular approach makes the system easy to understand, test, and extend.

The user's request reaches agents through the conversation context (shared via `sub_agents`). Agents pass data to each other through `output_key` and `{{variable}}` interpolation — for example, `topic_research_agent` stores its result in `blog_topic`, and `content_drafter_agent` reads it as `{{blog_topic}}`.

In [ ]:
from google.adk.agents import Agent
from google.adk.tools import google_search

# --- Topic Research Agent ---
topic_research_agent = Agent(
 name="topic_research_agent",
 model="gemini-2.5-flash",
 instruction="""
 You are a topic research expert. Based on the user's request,
 use search to find trending angles and select the SINGLE BEST specific blog post title.
 Output format: Just the title, nothing else.

 Example: "10 AI Tools That Save Small Businesses 20 Hours Per Week"
 """,
 tools=[google_search],
 output_key="blog_topic"
)

# --- Content Drafter Agent ---
content_drafter_agent = Agent(
 name="content_drafter_agent",
 model="gemini-2.5-flash",
 instruction="""
 You are a content writer. Write a blog post: {{blog_topic}}

 Create a draft (400-600 words) with:
 - Engaging introduction
 - At least 2 H2 headings
 - A conclusion section

 Output only the blog post in markdown format.
 """,
 tools=[],
 output_key="current_content"
)

# --- Quality Checker Agent ---
quality_checker_agent = Agent(
 name="quality_checker_agent",
 model="gemini-2.5-flash",
 instruction=f"""
 You are a content quality analyst. Analyze: {{{{current_content}}}}

 Your job:
 1. Count approximate word count
 2. Estimate readability score (60+ is good)
 3. Check for clear headings
 4. Check for conclusion section

 Use `calculate_content_quality_score` tool.

 Then:
 - IF overall_score >= 70, respond with: '{QUALITY_THRESHOLD_MET}'
 - ELSE, respond with: 'Quality score: [score]. Issues: [specific problems]'
 """,
 tools=[calculate_content_quality_score],
 output_key="quality_feedback"
)

# --- Content Improver Agent ---
content_improver_agent = Agent(
 name="content_improver_agent",
 model="gemini-2.5-flash",
 instruction=f"""
 Current content: {{{{current_content}}}}
 Feedback: {{{{quality_feedback}}}}
 
 - IF feedback is '{QUALITY_THRESHOLD_MET}': 
 1. Call the `exit_loop` tool to terminate the loop
 2. Then respond with: "Quality threshold met! Content approved."
 
 - ELSE: improve based on issues:
 * Expand if short (add examples, details, explanations)
 * Simplify if complex (shorter sentences, simpler words)
 * Add clear H2 headings if missing
 * Add a strong conclusion if missing
 
 Output the COMPLETE improved content in markdown.
 """,
 tools=[exit_loop],
 output_key="current_content"
)

print(" Core workflow agents created!")

In [ ]:
# --- Final Content Creators (Parallel) ---

blog_post_writer_agent = Agent(
 name="blog_post_writer_agent",
 model="gemini-2.5-flash",
 instruction="""
 You are a professional blog writer. Create the final polished blog post from: {{current_content}}

 Enhance it to be publication-ready:
 - Ensure 800-1200 words
 - Add engaging subheadings
 - Include actionable tips
 - Strong call-to-action

 Output only the final blog post in markdown.
 """,
 tools=[],
 output_key="final_blog_post"
)

social_media_creator_agent = Agent(
 name="social_media_creator_agent",
 model="gemini-2.5-flash",
 instruction="""
 You are a social media specialist. Create posts from: {{current_content}}

 Create:
 1. LinkedIn Post (150-200 words, professional)
 2. Twitter Thread (3 tweets, 280 chars each)
 3. Instagram Caption (100-150 words, with emojis and hashtags)

 Format with clear headers for each platform.
 """,
 tools=[],
 output_key="social_media_posts"
)

email_newsletter_writer_agent = Agent(
 name="email_newsletter_writer_agent",
 model="gemini-2.5-flash",
 instruction="""
 You are an email marketing specialist. Create a newsletter from: {{current_content}}

 Include:
 - Subject Line (compelling, 50-60 chars)
 - Preview Text (40-50 chars)
 - Body (300-400 words with CTA)

 Format with clear sections.
 """,
 tools=[],
 output_key="email_newsletter"
)

seo_metadata_agent = Agent(
 name="seo_metadata_agent",
 model="gemini-2.5-flash",
 instruction="""
 You are an SEO specialist. Generate metadata based on: {{current_content}}

 Create:
 1. Meta Title (50-60 chars)
 2. Meta Description (150-160 chars)
 3. URL Slug
 4. Focus Keyword
 5. 5 Related Keywords

 Format as readable markdown with bold labels, not JSON.
 """,
 tools=[],
 output_key="seo_metadata"
)

# --- Content Analyzer (Standalone) ---
content_analyzer_agent = Agent(
 name="content_analyzer_agent",
 model="gemini-2.5-flash",
 instruction="""
 You are a content analysis expert. Analyze the provided text.

 Use your tools to:
 1. Count words
 2. Calculate readability
 3. Generate 5 hashtags

 Provide a clear analysis report.
 """,
 tools=[count_words, calculate_readability_score, generate_hashtags]
)

print(" All specialist agents created!")

## 3. Assemble All Workflow Agents

### NEW CONCEPT: Composable Agent Systems

> **What are Composable Agent Systems?** 
> Composable agent systems are designed using modular, reusable components that can be combined in different ways to create complex workflows. Each component (agent, tool, workflow) is self-contained and well-defined, allowing it to be easily plugged into larger systems without modification.
>
> **Key Principles**:
> 1. **Modularity**: Each component has a single, clear responsibility
> 2. **Reusability**: Components can be used in multiple workflows
> 3. **Composability**: Components combine seamlessly via standard interfaces
> 4. **Independence**: Components don't depend on implementation details of others
>
> **How It Works**:
> ```python
> # Build small components
> agent1 = Agent(...)
> agent2 = Agent(...)
> 
> # Compose into workflows
> workflow1 = SequentialAgent(sub_agents=[agent1, agent2])
> 
> # Compose workflows into larger systems
> master = SequentialAgent(sub_agents=[workflow1, workflow2])
> ```
>
> **Benefits**:
> - **Testable**: Test each component independently
> - **Maintainable**: Update one component without affecting others
> - **Scalable**: Add new capabilities by composing new components
> - **Flexible**: Rearrange components to create new workflows
> - **Understandable**: Clear hierarchy and data flow
>
> **In This Capstone**:
> - 9 specialist agents
> - 3 workflow types: Sequential, Loop, Parallel (from Parts 4-6)
> - All composed into one master system
>
> **Reference**: [Multi-Agent Systems](https://google.github.io/adk-docs/agents/)

---

Now we compose our specialists into powerful, multi-step workflow agents using Sequential, Loop, and Parallel patterns. This is where all the patterns from Parts 1-6 come together!

**Key design decisions:**
- **No final packager**: Each parallel agent stores output via `output_key` — the results are already accessible individually
- **max_iterations=2**: Keeps execution fast while still allowing quality improvement

In [ ]:
from google.adk.agents import SequentialAgent, LoopAgent, ParallelAgent

# --- Sequential: Research and Draft ---
research_and_draft_workflow = SequentialAgent(
 name="research_and_draft_workflow",
 sub_agents=[topic_research_agent, content_drafter_agent]
)

# --- Loop: Quality Improvement ---
quality_improvement_loop = LoopAgent(
 name="quality_improvement_loop",
 sub_agents=[quality_checker_agent, content_improver_agent],
 max_iterations=2
)

# --- Parallel: Multi-Channel Content Creation ---
parallel_content_creation = ParallelAgent(
 name="parallel_content_creation",
 sub_agents=[
 blog_post_writer_agent,
 social_media_creator_agent,
 email_newsletter_writer_agent,
 seo_metadata_agent
 ]
)

# --- Full Content Workflow ---
# No intake agent needed — the caller sets content_brief in session state.
# No final packager needed — each parallel agent stores its output via output_key.
full_content_workflow = SequentialAgent(
 name="full_content_workflow",
 sub_agents=[
 research_and_draft_workflow,
 quality_improvement_loop,
 parallel_content_creation,
 ]
)

print(" All workflow agents assembled!")
print("\n Workflow Structure:")
print(" 1. Sequential → Research + Draft")
print(" 2. Loop → Quality check + Improve (up to 2 iterations)")
print(" 3. Parallel → Blog + Social + Email + SEO (concurrent)")

## 4. Build the Master Orchestrator

### NEW CONCEPT: Hierarchical Orchestration

> **What is Hierarchical Orchestration?** 
> Hierarchical orchestration is an architectural pattern where a top-level "master" agent manages multiple layers of sub-systems, which can themselves be simple agents or complex multi-agent workflows. The master agent delegates tasks based on intent, routing requests to the appropriate subsystem without needing to know their internal implementation details.
>
> **Architecture Layers**:
> ```
> Layer 1: Master Orchestrator (routing & delegation)
> ↓
> Layer 2: Sub-Workflows (Sequential, Loop, Parallel workflows)
> ↓
> Layer 3: Specialist Agents (individual task experts)
> ↓
> Layer 4: Tools (custom functions, APIs, built-in tools)
> ```
>
> **How It Works**:
> ```python
> # Layer 3: Specialists
> agent1 = Agent(tools=[custom_tool1])
> agent2 = Agent(tools=[custom_tool2])
> 
> # Layer 2: Workflows
> workflow = SequentialAgent(sub_agents=[agent1, agent2])
> 
> # Layer 1: Master (uses sub_agents for event visibility)
> master = Agent(
>     sub_agents=[workflow, analyzer_agent],
>     tools=[memory_tool]
> )
> ```
>
> **Key Characteristics**:
> - **Delegation**: Master routes requests without executing tasks itself
> - **Abstraction**: Master doesn't know internal workflow details
> - **Flexibility**: Can handle both simple and complex requests
> - **Scalability**: Easy to add new sub-workflows
> - **Single Interface**: Users interact with one entry point
>
> **Why `sub_agents` instead of `AgentTool`?**
>
> In Part 3, we used `AgentTool` to wrap agents as tools. Here we use `sub_agents` instead because:
> - **Event visibility**: With `sub_agents`, all inner events propagate to the Runner — we can see each parallel agent's output as it completes
> - **Shared context**: Sub-agents share the same session and invocation context
> - **Stateful workflows**: The content pipeline is a complex, stateful process that benefits from shared state
>
> See the comparison note in Part 3 for a detailed breakdown of AgentTool vs sub_agents.
>
> **Reference**: [Multi-Agent Systems](https://google.github.io/adk-docs/agents/)

---

This is our final, user-facing "command center." Its only job is to delegate tasks — it transfers control to the full content workflow for creation requests, or to the content analyzer for analysis requests. It also has a memory tool for cross-session context.

In [ ]:
from google.adk.tools import preload_memory_tool

# The orchestrator uses sub_agents (not AgentTool) so inner events propagate.
# The LLM transfers control to the appropriate sub-agent based on user intent.
master_orchestrator_agent = Agent(
 name="master_orchestrator_agent",
 model="gemini-2.5-flash",
 instruction="""
 You are the Master Content Creation Studio orchestrator. Delegate tasks to specialists.

 Past conversations from long-term memory are automatically loaded before each turn.
 Use this context to provide continuity across sessions.

 - For FULL content creation (topic research -> draft -> improve -> multi-channel content),
   transfer to `full_content_workflow`. Pass the complete user request with topic, audience, tone, and keywords.

 - For ANALYZING existing text (readability, word count, hashtags),
   transfer to `content_analyzer_agent`.

 Always delegate to the appropriate agent.
 """,
 sub_agents=[full_content_workflow, content_analyzer_agent],
 tools=[preload_memory_tool.PreloadMemoryTool()],
 before_agent_callback=before_agent_callback,
 after_agent_callback=after_agent_callback,
 before_model_callback=before_model_callback,
 after_model_callback=after_model_callback,
)

print(" Master Orchestrator created!")
print("\n Capabilities:")
print("   Full content workflow (research -> draft -> improve -> multi-channel)")
print("   Quick content analysis (words, readability, hashtags)")
print("   Callbacks: guardrails, timing, output metrics")
print("   Memory: auto-loads past conversations via PreloadMemoryTool")

## 5. The Grand Finale — A Full Conversation

Finally, we run a full, multi-turn conversation with our "Autonomous Content Creation Studio." This demonstrates its ability to handle complex initial requests and then switch to simpler, interactive follow-up tasks.

This is complete AI agent architecture in action!

In [ ]:
from google.adk.sessions import InMemorySessionService, Session
from google.adk.runners import Runner
from google.adk.artifacts import InMemoryArtifactService
from google.adk.memory import InMemoryMemoryService
from google.genai.types import Content, Part

session_service = InMemorySessionService()
artifact_service = InMemoryArtifactService()
memory_service = InMemoryMemoryService()
user_id = "adk_content_creator_001"

async def run_agent_query(agent: Agent, query: str, session: Session, user_id: str):
 """Initializes a runner and executes a query for a given agent and session."""
 print(f"\n{'='*70}")
 print(f" Running query for agent: '{agent.name}'")
 print(f"{'='*70}\n")

 runner = Runner(
 agent=agent,
 session_service=session_service,
 artifact_service=artifact_service,
 memory_service=memory_service,
 app_name=agent.name,
 )

 final_response = ""
 try:
 async for event in runner.run_async(
 user_id=user_id,
 session_id=session.id,
 new_message=Content(parts=[Part(text=query)], role="user")
 ):
 if event.is_final_response() and event.content and event.content.parts:
 text = event.content.parts[0].text
 if text:
 final_response = text
 except Exception as e:
 final_response = f"An error occurred: {e}"

 print(f"\n{'='*70}")
 print(" FINAL RESPONSE:")
 print(f"{'='*70}")
 display(Markdown(final_response))
 print(f"{'='*70}\n")

 return final_response

In [ ]:
async def run_capstone_project():
 """Run the complete Content Creation Studio system"""

 session = await session_service.create_session(
 app_name=master_orchestrator_agent.name,
 user_id=user_id,
 )

 print("\n" + ""*35)
 print(" CONTENT CREATION STUDIO - CAPSTONE PROJECT")
 print(""*35 + "\n")

 # --- Query 1: Full Content Creation ---
 query1 = """
 Create a complete content package for:
 - Topic: Productivity hacks using AI for remote workers
 - Target Audience: Remote professionals and digital nomads
 - Tone: Conversational and helpful
 - Keywords: AI productivity, remote work, automation tools
 """
 print(f" USER REQUEST 1:\n{query1}\n")
 await run_agent_query(master_orchestrator_agent, query1, session, user_id)

 # --- Query 2: Analyze Specific Content ---
 sample_text = """
 Remote work has transformed how we think about productivity. With AI tools,
 professionals can automate repetitive tasks and focus on creative work.
 """
 query2 = f"Can you analyze this text snippet:\n\n{sample_text}"
 print(f" USER REQUEST 2:\n{query2}\n")
 await run_agent_query(master_orchestrator_agent, query2, session, user_id)

 print("\n" + ""*35)
 print(" CAPSTONE PROJECT COMPLETE!")
 print(""*35 + "\n")

# Run the complete system
await run_capstone_project()

## Expected Complete Output

The system will demonstrate:

### 1. Full Content Creation Workflow:
- **Research trending angle** (search)
- **Draft initial content**
- **Check quality** (loop iteration 1)
- **Improve content** (loop iteration 2)
- **Final quality check** (meets threshold, exit loop)
- **Create blog, social, email, SEO in parallel**

### 2. Callbacks in Action:
- `before_agent_callback` logs agent start with timestamp
- `after_agent_callback` logs agent completion with elapsed time
- `before_model_callback` checks for blocked topics (guardrail)
- `after_model_callback` logs output word count

### 3. Quick Content Analysis:
- Word count
- Readability score
- Generated hashtags

## Complete Workshop Recap: Parts 1-8

## Complete Learning Journey

### Part 1 -- The Foundation 
**Concepts**: Agent, Built-in Tools, Session, SessionService, Runner 
**What We Built**: Single agent with google_search 
**Key Learning**: Basic agent creation and execution 
 [Agents](https://google.github.io/adk-docs/agents/) | [Runtime](https://google.github.io/adk-docs/runtime/)

### Part 2 -- Custom Skills 
**Concepts**: Custom Function Tools, Tool Signatures, Specialist Agent Pattern 
**What We Built**: Content analyzer with custom Python tools 
**Key Learning**: Creating tools from Python functions 
 [Custom Tools](https://google.github.io/adk-docs/tools/)

### Part 3 -- Teamwork 
**Concepts**: AgentTool, Orchestrator Pattern, Multi-Agent Coordination 
**What We Built**: Orchestrator managing specialist team 
**Key Learning**: Agents as tools, delegation 
 [Multi-Agent Systems](https://google.github.io/adk-docs/agents/)

### Part 4 -- Ordered Tasks 
**Concepts**: SequentialAgent, output_key, Variable Interpolation, State Passing 
**What We Built**: Research -> Draft -> Format pipeline 
**Key Learning**: Sequential workflows with automatic state passing 
 [Workflow Agents](https://google.github.io/adk-docs/agents/)

### Part 5 -- Iteration 
**Concepts**: LoopAgent, ToolContext, tool_context.actions.escalate, Critique-Refine Pattern 
**What We Built**: Quality improvement loop (draft -> critique -> improve -> repeat) 
**Key Learning**: Iterative refinement until quality thresholds met 
 [Workflow Agents](https://google.github.io/adk-docs/agents/) | [Tool Context](https://google.github.io/adk-docs/tools/)

### Part 6 -- Efficiency 
**Concepts**: ParallelAgent, Fan-Out/Fan-In Pattern, output_key for state passing 
**What We Built**: Concurrent multi-channel content creation 
**Key Learning**: Parallel execution for massive efficiency gains 
 [Workflow Agents](https://google.github.io/adk-docs/agents/)

### Part 7 -- Callbacks, Context & Memory
**Concepts**: Callbacks, Sessions/State, Artifacts, Memory, PreloadMemoryTool
**What We Built**: Callback-enhanced agents with state management and cross-session memory
**Key Learning**: Runtime control, data persistence layers, long-term memory

### Part 8 -- The Capstone 
**Concepts**: Hierarchical Orchestration, Composable Systems, End-to-End Autonomous Workflows 
**What We Built**: Complete autonomous Content Creation Studio combining ALL patterns from 9 parts 
**Key Learning**: How to architect multi-agent systems that combine every pattern

---

## What This Capstone Demonstrates

### 1. Hierarchical Orchestration
We built a master agent that sits at the top of a hierarchy, capable of delegating to both simple tools and entire complex sub-workflows. This provides:
- Single user-facing interface
- Clean separation of concerns
- Easy extensibility

### 2. Callbacks for Guardrails and Observability
Four callback types provide control over agent execution:
- **before_agent_callback**: Logs agent start, tracks timing
- **after_agent_callback**: Logs completion with elapsed time
- **before_model_callback**: Content safety guardrail -- blocks prohibited topics before they reach the model
- **after_model_callback**: Logs output metrics (word count)

### 3. Memory for Cross-Session Context
PreloadMemoryTool automatically loads past conversations before each turn, enabling the orchestrator to provide continuity across sessions.

### 4. Composable Systems
We proved the power of modular design. Every component built throughout the series was reused as a plug-and-play part in our final application:
- **From Part 1-2**: Basic agents with custom tools
- **From Part 3**: AgentTool wrapper for delegation
- **From Part 4**: Sequential workflows for ordered steps
- **From Part 5**: Loop workflows for quality improvement
- **From Part 6**: Parallel workflows for efficiency
- **From Part 7**: Callbacks, state management, and memory

### 5. Architecture Best Practices
- Callbacks for guardrails and observability
- State management via `output_key` across workflow stages
- Memory for cross-session context
- Modular, testable components
- Clear execution flow
- Scalable architecture

---

## Architecture Summary

```
Layer 1: Master Orchestrator (+ callbacks + memory)
 |-- Full Content Workflow (AgentTool)
 |-- Content Analyzer (AgentTool)

Layer 2: Full Content Workflow
 |-- Research & Draft Workflow (Sequential)
 |-- Quality Improvement Loop (Loop, max_iterations=2)
 |-- Parallel Content Creation (Parallel)

Layer 3: Specialist Agents (9 total)
 Each with specific tools and expertise
 Each stores output via output_key

Layer 4: Tools (Custom Functions + Built-in)
 Content analysis, quality scoring, loop control, google_search
```

### The ADK Agent Hierarchy You've Mastered

![ADK Agent Types](https://google.github.io/adk-docs/assets/agent-types.png)

*Source: [Google ADK Agents Documentation](https://google.github.io/adk-docs/agents/)*

You have now used **all three** agent types:
- **LLM Agents**: topic_research_agent, content_drafter_agent, all specialists (Parts 1-3)
- **Workflow Agents**: SequentialAgent (Part 4), LoopAgent (Part 5), ParallelAgent (Part 6)
- **Custom Agents**: Not covered in this workshop, but available for advanced use cases

---

## From Concepts to Capstone

**You started with**: A simple agent that searches and suggests topics 
**You ended with**: A complete autonomous content studio with callbacks, memory, and multi-channel parallel output

This is the power of composable, hierarchical agent architecture!

## Congratulations!

You've completed the entire Google ADK Playbook! You now have the knowledge and practical experience to build sophisticated AI agent systems.

### What You Can Do Now:
- Build multi-agent systems for any domain
- Create custom tools tailored to your needs
- Design complex workflows (sequential, iterative, parallel)
- Orchestrate teams of specialist agents
- Build complete AI applications

### Next Steps:
1. **Experiment**: Modify this capstone project for your own use case
2. **Extend**: Add new agents and tools specific to your domain
3. **Deploy**: Continue to Part 9 to deploy your system to Google Cloud
4. **Share**: Build amazing things and share your learnings!

---

## Additional Resources

- **Official Docs**: https://google.github.io/adk-docs
- **GitHub**: https://github.com/google/adk
- **Google AI Studio**: https://aistudio.google.com

---

**Thank you for completing the Google ADK Playbook!** 

We can't wait to see what you build next!